In [ ]:
import requests
import json

# 全国版の地下空間抽出クエリ
overpass_url = "http://overpass-api.de/api/interpreter"
overpass_query = """
[out:json][timeout:300];
area["name"="日本"]->.japan;
(
  way["railway"]["tunnel"="yes"](area.japan);
  way["highway"~"pedestrian|footway"]["tunnel"="yes"](area.japan);
  way["location"="underground"](area.japan);
);
out body;
>;
out skel qt;
"""

print("Googleのサーバーが全国の地下データを収集中です...（2~3分かかります）")
response = requests.post(overpass_url, data={'data': overpass_query})
data = response.json()

with open('nationwide_shadow_network.json', 'w') as f:
    json.dump(data, f)

print("完了！ nationwide_shadow_network.json が生成されました。")

Googleのサーバーが全国の地下データを収集中です...（2~3分かかります）
完了！ nationwide_shadow_network.json が生成されました。


In [ ]:
import json

# 1. 保存した生データを読み込む
with open('nationwide_shadow_network.json', 'r') as f:
    osm_data = json.load(f)

# 2. OSM形式からGeoJSON形式へ翻訳（シンプルな変換ロジック）
nodes = {n['id']: (n['lon'], n['lat']) for n in osm_data['elements'] if n['type'] == 'node'}
features = []

for el in osm_data['elements']:
    if el['type'] == 'way' and 'nodes' in el:
        coords = [nodes[node_id] for node_id in el['nodes'] if node_id in nodes]
        if len(coords) > 1:
            feature = {
                "type": "Feature",
                "properties": el.get('tags', {}),
                "geometry": {
                    "type": "LineString",
                    "coordinates": coords
                }
            }
            features.append(feature)

geojson_out = {
    "type": "FeatureCollection",
    "features": features
}

# 3. ファイルを保存（名前を変えて保存します）
output_file = 'nationwide_shadow_net_FINAL.geojson'
with open(output_file, 'w') as f:
    json.dump(geojson_out, f)

print(f"変換完了！ '{output_file}' をダウンロードして geojson.io に入れてみてください。")
print(f"抽出されたライン数: {len(features)}")

変換完了！ 'nationwide_shadow_net_FINAL.geojson' をダウンロードして geojson.io に入れてみてください。
抽出されたライン数: 21480
